In [1]:
import random

In [2]:
import numpy as np

In [3]:
import matplotlib
matplotlib.use("Agg")

In [4]:
import matplotlib.pyplot as plt

In [5]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from sklearn.datasets import make_moons

In [7]:
!pip install micrograd


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\suraj\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
from micrograd.engine import Value
from micrograd.nn import MLP

ModuleNotFoundError: No module named 'micrograd'

In [9]:
import sys
!{sys.executable} -m pip install micrograd scikit-learn


  Using cached micrograd-0.1.0-py3-none-any.whl.metadata (2.6 kB)
Using cached micrograd-0.1.0-py3-none-any.whl (4.9 kB)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from micrograd.engine import Value
from micrograd.nn import MLP

In [11]:
np.random.seed(1337)
random.seed(1337)

In [12]:
X, y = make_moons(n_samples = 100, noise = 0.1)

In [13]:
y = y * 2 - 1

In [14]:
model = MLP(2, [16,16,1])
print(model)

MLP of [Layer of [ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2), ReLUNeuron(2)], Layer of [ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16), ReLUNeuron(16)], Layer of [LinearNeuron(16)]]


In [15]:
print("number of parameters:", len(model.parameters()))

number of parameters: 337


In [18]:
def loss(batch_size = None):
    
    if batch_size is None:
        Xb, yb = X, y
    else:
        ri = np.random.permutation(X.shape[0])[:batch_size]
        Xb, yb = X[ri], y[ri]

    inputs = []

    for xrow in Xb:
        one_input = []
        for number in xrow:
            one_input.append(Value(number))
        inputs.append(one_input)

    scores = []

    for one_input in inputs:
        score = model(one_input)
        scores.append(score)

    losses = []

    for yi, scorei in zip(yb, scores):
        loss_i = 1 - yi * scorei
        loss_i = loss_i.relu()
        losses.append(loss_i)

    data_loss = sum(losses) * (1.0 / len(losses))

    alpha = 1e-4

    reg_loss = 0

    for p in model.parameters():
        reg_loss += p * p

    reg_loss = alpha * reg_loss

    total_loss = data_loss + reg_loss

    accuracy = []

    for yi, scorei in zip(yb, scores):

        if yi > 0:
            actual_class = 1
        else: 
            actual_class = -1

        if scorei.data > 0:
            predicted_class = 1
        else:
            predicted_class = -1

        if actual_class == predicted_class:
            accuracy.append(True)
        else:
            accuracy.append(False)
    return total_loss, sum(accuracy) / len(accuracy)

In [19]:
for k in range(100):
    total_loss, acc = loss()

    model.zero_grad()
    total_loss.backward()


    learning_rate = 1.0 - 0.9 * k / 100

    for p in model.parameters():
        p.data -= learning_rate * p.grad

    if k % 10 == 0 or k == 99:
        print(f"step {k:3d}  loss {total_loss.data:.4f}  accuracy {acc * 100:.1f}%")

step   0  loss 0.8958  accuracy 50.0%
step  10  loss 0.2451  accuracy 91.0%
step  20  loss 0.1898  accuracy 91.0%
step  30  loss 0.1173  accuracy 95.0%
step  40  loss 0.0602  accuracy 100.0%
step  50  loss 0.0988  accuracy 96.0%
step  60  loss 0.0326  accuracy 99.0%
step  70  loss 0.0142  accuracy 100.0%
step  80  loss 0.0123  accuracy 100.0%
step  90  loss 0.0110  accuracy 100.0%
step  99  loss 0.0110  accuracy 100.0%


In [20]:
h = 0.25
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Xmesh = np.c_[xx.ravel(), yy.ravel()]          # flatten grid into a list of points
 
inputs = [list(map(Value, xrow)) for xrow in Xmesh]
scores = list(map(model, inputs))
Z = np.array([s.data > 0 for s in scores])     # True/False -> predicted class
Z = Z.reshape(xx.shape)
 
fig = plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)   # colored regions
plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.Spectral, edgecolors="k")
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
plt.title("micrograd MLP — make_moons decision boundary")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.savefig("moons_decision_boundary.png", dpi=130, bbox_inches="tight")
print("saved: moons_decision_boundary.png")

saved: moons_decision_boundary.png
